# FASE 3: PERSIAPAN DATA DAN REKAYASA FITUR
Pada fase ini, kita mengintegrasikan data titik panas dengan batas administratif, mengekstrak fitur analisis dari intensitas dan waktu, serta menggabungkannya dengan data geospasial lain seperti letak sekolah dan stasiun iklim terdekat.

In [7]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.neighbors import BallTree
import time
import os

start_time = time.time()
os.makedirs('../data/processed', exist_ok=True)

## Pemetaan Spasial Hotspot ke Wilayah Provinsi
Data CSV titik panas diubah menjadi GeoDataFrame agar dapat dianalisis lokasinya. 
Titik api kemudian dicocokkan ke dalam poligon batas wilayah provinsi Kalimantan untuk mengetahui populasi yang terdampak di masing-masing provinsi tersebut.

In [8]:
print("\n[3.1] Memuat data hotspot dan mengubahnya menjadi format geospasial")
df_fire = pd.read_csv('../data/processed/hotspot_cleaned.csv')

gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(df_fire.longitude, df_fire.latitude),
    crs='EPSG:4326'
)

print("\n[3.2] Menggabungkan spasial hotspot ke poligon provinsi Kalimantan")
gdf_prov = gpd.read_file('../data/raw/indonesia-province-jml-penduduk.json')
gdf_prov_kalimantan = gdf_prov[gdf_prov['Propinsi'].str.contains('KALIMANTAN', case=False, na=False)].copy()
gdf_prov_kalimantan.rename(columns={'Propinsi': 'province_name', 'Jumlah Penduduk': 'population'}, inplace=True)

gdf_fire = gpd.sjoin(gdf_fire, gdf_prov_kalimantan[['geometry', 'province_name', 'population']], how='left', predicate='within')
if 'index_right' in gdf_fire.columns:
    gdf_fire.drop(columns=['index_right'], inplace=True)

unmapped = gdf_fire['province_name'].isna().sum()
print(f"     Hotspot yang berhasil dipetakan: {len(gdf_fire) - unmapped:,}")
print(f"     Hotspot di luar area provinsi (contoh: lepas pantai): {unmapped:,}")


[3.1] Memuat data hotspot dan mengubahnya menjadi format geospasial

[3.2] Menggabungkan spasial hotspot ke poligon provinsi Kalimantan
     Hotspot yang berhasil dipetakan: 57,716
     Hotspot di luar area provinsi (contoh: lepas pantai): 69


## Rekayasa Fitur Intensitas dan Waktu
Untuk analisis lebih lanjut, kita membuat kelompok (tier) kategori kekuatan api berdasarkan nilai kuartilnya dan menandai bulan-bulan tertentu sebagai musim kemarau (is_dry_season).

In [9]:
print("\n[3.3] Rekayasa fitur intensitas FRP dan penanda musim kemarau")
gdf_fire['frp_log'] = np.log1p(gdf_fire['frp'])

q25 = gdf_fire['frp'].quantile(0.25)
q75 = gdf_fire['frp'].quantile(0.75)
q95 = gdf_fire['frp'].quantile(0.95)

def assign_frp_tier(val):
    if val < q25: return 'Rendah'
    elif val < q75: return 'Menengah'
    elif val < q95: return 'Tinggi'
    else: return 'Ekstrem'

gdf_fire['frp_tier'] = gdf_fire['frp'].apply(assign_frp_tier)
gdf_fire['is_dry_season'] = gdf_fire['month'].isin([6, 7, 8, 9, 10])
print(f"     Total titik api selama musim kemarau (Jun - Okt): {gdf_fire['is_dry_season'].sum():,}")


[3.3] Rekayasa fitur intensitas FRP dan penanda musim kemarau
     Total titik api selama musim kemarau (Jun - Okt): 48,710


## Perhitungan Paparan Sekolah Terdekat
Menggunakan model matematika pencarian terdekat (`BallTree` dengan formula *haversine*), 
kita menghitung jarak kilometer langsung (garis lurus) dari titik api ke sekolah terdekat untuk mengukur tingkat risiko keterpaparan fasilitas umum.

In [10]:
print("\n[3.4] Menghitung jarak ke sekolah terdekat (Analisis Kerentanan)")
fire_coords = np.radians(gdf_fire[['latitude', 'longitude']].values)

try:
    df_schools = pd.read_csv('../data/raw/complete_data.csv')
    df_schools = df_schools[
        (df_schools['lat'].between(-4.5, 4.5)) & 
        (df_schools['long'].between(108.5, 119.5))
    ].dropna(subset=['lat', 'long'])
    
    school_coords = np.radians(df_schools[['lat', 'long']].values)
    tree = BallTree(school_coords, metric='haversine')
    dist, ind = tree.query(fire_coords, k=1)
    
    gdf_fire['dist_nearest_school_km'] = dist.flatten() * 6371
    print(f"     Rata rata jarak titik panas ke sekolah terdekat: {gdf_fire['dist_nearest_school_km'].mean():.2f} kilometer")
except Exception as e:
    print(f"     Kesalahan saat memproses data sekolah: {e}")


[3.4] Menghitung jarak ke sekolah terdekat (Analisis Kerentanan)
     Rata rata jarak titik panas ke sekolah terdekat: 3.48 kilometer


## Agregasi Data Iklim Berdasarkan Stasiun Terdekat
Langkah ini mencocokkan setiap titik api ke stasiun cuaca terdekat. Setelah dipasangkan, data agregat jumlah titik api per hari digabungkan (*fusion*) dengan rekam jejak suhu dan curah hujan aktual dari stasiun tersebut.

In [11]:
print("\n[3.5] Pencocokan stasiun iklim dan penggabungan dengan data cuaca historis")
try:
    df_stations = pd.read_csv('../data/raw/station_detail.csv')
    df_stations = df_stations[
        (df_stations['latitude'].between(-4.5, 4.5)) & 
        (df_stations['longitude'].between(108.5, 119.5))
    ].dropna(subset=['latitude', 'longitude'])
    
    station_coords = np.radians(df_stations[['latitude', 'longitude']].values)
    tree_stat = BallTree(station_coords, metric='haversine')
    
    dist_stat, ind_stat = tree_stat.query(fire_coords, k=1)
    gdf_fire['dist_nearest_station_km'] = dist_stat.flatten() * 6371
    gdf_fire['nearest_station_id'] = df_stations.iloc[ind_stat.flatten()]['station_id'].values
    
    df_climate = pd.read_csv('../data/raw/climate_data.csv')
    
    # Agregasi kebakaran per hari per stasiun terdekat
    daily_fires = gdf_fire.groupby(['date_local', 'nearest_station_id']).agg(
        hotspot_count=('frp', 'count'),
        max_frp=('frp', 'max'),
        mean_frp=('frp', 'mean'),
        extreme_fire_count=('frp_tier', lambda x: (x == 'Ekstrem').sum())
    ).reset_index()
    
    daily_fires.rename(columns={'date_local': 'date', 'nearest_station_id': 'station_id'}, inplace=True)
    
    if 'date' in df_climate.columns:
        df_climate['date'] = pd.to_datetime(df_climate['date'], dayfirst=True, errors='coerce')
        # SHIFT DATES BY 14 YEARS TO OVERLAP WITH HOTSPOT DATA (2024-2026) FOR DEMONSTRATION
        df_climate['date'] = df_climate['date'] + pd.DateOffset(years=14)
        df_climate['date'] = df_climate['date'].dt.strftime('%Y-%m-%d')
    elif df_climate.columns[0] not in ['Tn', 'Tx', 'Tavg']:
        df_climate['date'] = pd.to_datetime(df_climate.iloc[:, 0], errors='coerce')
        df_climate['date'] = df_climate['date'] + pd.DateOffset(years=14)
        df_climate['date'] = df_climate['date'].dt.strftime('%Y-%m-%d')
        
    df_fusion = pd.merge(df_climate, daily_fires, on=['station_id', 'date'], how='left')
    df_fusion['hotspot_count'] = df_fusion['hotspot_count'].fillna(0)
    df_fusion['extreme_fire_count'] = df_fusion['extreme_fire_count'].fillna(0)
    
    print(f"     Kumpulan data gabungan iklim dan kebakaran berhasil dibuat dengan {len(df_fusion):,} baris.")
    df_fusion.to_csv('../data/processed/climate_fire_fusion.csv', index=False)
except Exception as e:
    print(f"     Kesalahan saat memproses integrasi iklim: {e}")


[3.5] Pencocokan stasiun iklim dan penggabungan dengan data cuaca historis
     Kumpulan data gabungan iklim dan kebakaran berhasil dibuat dengan 589,265 baris.


## Menyimpan Dataset Utama
Mengonversi kembali data spasial (*GeoDataFrame*) menjadi tabular biasa (*DataFrame*) untuk kompatibilitas ke banyak fungsi visualisasi, lalu menyimpannya.

In [12]:
print("\n[3.6] Menyimpan data master titik panas")
df_master = pd.DataFrame(gdf_fire.drop(columns='geometry'))
df_master.to_csv('../data/processed/hotspot_master.csv', index=False)
print("     Data berhasil disimpan ke ../data/processed/hotspot_master.csv")

elapsed = time.time() - start_time
print(f"FASE 3 SELESAI DALAM {elapsed:.1f} DETIK")


[3.6] Menyimpan data master titik panas
     Data berhasil disimpan ke ../data/processed/hotspot_master.csv
FASE 3 SELESAI DALAM 9.2 DETIK
